# 01 - Hello World: 基本設定與 LLM 呼叫

本筆記本示範如何在此框架中進行最基本的操作。

## 本筆記本涵蓋內容

本範例將逐步示範以下三個核心操作：

1. **載入設定（Config Loading）**：使用 Hydra 從 `config/config.yaml` 載入專案設定，並透過 `OmegaConf` 檢視設定內容。
2. **設定 Logger**：透過 `setup_logging` 初始化日誌系統，並使用 `get_logger` 取得具名 logger 記錄訊息。
3. **初始化並呼叫 LLM Client**：使用 `LLMClient` 連接 LLM 服務，發送對話請求並解析回應結果。

> **前置條件**：請確保已安裝相依套件（`uv sync`），並已在 `config/config.yaml` 中填入正確的 API 設定。

In [ ]:
# 標準函式庫
import sys
import os

# 將專案根目錄加入 Python 路徑
sys.path.insert(0, os.path.abspath(".."))

# Hydra / OmegaConf
from omegaconf import OmegaConf

# 專案模組
from app.utils.config import init_config, get_config, get_llm_config, reset_config
from app.logger import setup_logging, get_logger
from llm_service import LLMClient, LLMResponse

print("匯入完成。")

## 載入設定

使用 `init_config()` 初始化全域設定單例（singleton）。此函式會讀取 `config/config.yaml`，並支援多環境（dev / test / stg / prod）切換。

初始化後即可透過 `get_config()` 在任何地方取得相同的設定物件，無需重複傳遞。

In [ ]:
# 初始化設定（若需重置請先呼叫 reset_config()）
reset_config()  # 確保乾淨的初始狀態
cfg = init_config(config_path="../config", config_name="config")

# 使用 OmegaConf 將設定以 YAML 格式輸出，便於檢視
print("=== 目前設定內容 ===")
print(OmegaConf.to_yaml(cfg))

## 設定 Logger

使用 `setup_logging(cfg)` 根據設定中的 `logging` 區塊初始化日誌系統，包含日誌等級、輸出格式與檔案路徑等。

初始化完成後，透過 `get_logger(name)` 取得具名 logger，即可在程式各處統一記錄日誌訊息。

In [ ]:
# 根據設定初始化日誌系統
setup_logging(cfg)

# 取得本筆記本專用的 logger
logger = get_logger("01_hello_world")

# 記錄一則測試訊息
logger.info("Logger 初始化完成，開始執行 Hello World 範例。")

print("Logger 設定完畢。")

## 初始化 LLM Client

使用 `get_llm_config()` 取得設定中 `llm` 區塊的設定值（包含 API 端點、模型名稱、timeout 等），並將其傳入 `LLMClient` 建立客戶端實例。

`LLMClient` 內部使用 `httpx` 進行 HTTP 請求，並支援自動重試機制。

In [ ]:
# 取得 LLM 相關設定
llm_config = get_llm_config()

print("=== LLM 設定 ===")
print(OmegaConf.to_yaml(llm_config))

# 建立 LLM Client 實例
client = LLMClient(llm_config)

logger.info("LLMClient 初始化完成。")
print("LLMClient 初始化完畢。")

## 呼叫 LLM

使用 `client.chat()` 發送對話請求。此方法接受 `system_prompt` 與 `user_prompt` 兩個參數，並回傳 `LLMResponse` 物件。

`LLMResponse` 包含以下欄位：
- `content`：模型的文字回應
- `model`：實際使用的模型名稱
- `usage`：token 使用量統計
- `latency_ms`：請求延遲時間（毫秒）

> **注意**：此步驟需要一個正在運行的 LLM 服務（例如本地的 Ollama 或遠端 API）。若服務未啟動，將會捕捉例外並顯示錯誤訊息，不會中斷程式執行。

In [ ]:
system_prompt = "你是一個有用的助手。"
user_prompt = "請用一句話介紹自己。"

try:
    logger.info("正在發送 LLM 請求...")
    response: LLMResponse = client.chat(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
    )

    print("=== LLM 回應 ===")
    print(f"內容     : {response.content}")
    print(f"模型     : {response.model}")
    print(f"Token 用量: {response.usage}")
    print(f"延遲     : {response.latency_ms:.1f} ms")

    logger.info(
        "LLM 呼叫成功。model=%s, latency_ms=%.1f",
        response.model,
        response.latency_ms,
    )

except Exception as e:
    logger.warning("LLM 呼叫失敗（服務可能尚未啟動）：%s", e)
    print(f"[警告] LLM 呼叫失敗：{e}")
    print("請確認 LLM 服務已啟動，並在 config/config.yaml 中填入正確的 API 設定。")

## 小結

本筆記本示範了此框架的三個基本操作：

| 步驟 | 函式 / 類別 | 說明 |
|------|------------|------|
| 載入設定 | `init_config()` / `get_config()` | 使用 Hydra 載入 YAML 設定，支援多環境切換 |
| 設定 Logger | `setup_logging()` / `get_logger()` | 初始化結構化日誌系統 |
| 呼叫 LLM | `LLMClient.chat()` | 發送對話請求並解析回應 |

### 後續步驟

- 查看 `02_mlflow_logging.ipynb` 了解如何整合 MLflow 進行實驗追蹤。
- 查看 `03_langgraph_workflow.ipynb` 了解如何使用 LangGraph 建構多步驟工作流程。
- 修改 `config/config.yaml` 以連接您自己的 LLM 服務端點。